# EduVision_DV – Data Cleaning & Validation Notebook

## Objectives
This notebook executes the data cleaning pipeline across all four core datasets:
1. **QS World University Rankings 2025**
2. **THE World University Rankings 2023 / 2024**
3. **World Bank Education Statistics**
4. **World Bank Country Metadata**

### Guiding Principles & Rules
- **Non-Destructive**: Never modify raw data in `data/raw/`.
- **Preserve Legitimate Missing Data**: Do not replace NaNs with zeros or dummy values.
- **No Aggressive Merging**: Do not merge universities or perform fuzzy matching at this stage.
- **Detailed Logging**: Document all column renames, string parsing, and duplicate removals.
- **Clean Export**: Save standardized datasets into `data/cleaned/`.

In [ ]:
import os
import re
import pandas as pd
import numpy as np

raw_dir = r'../data/raw'
cleaned_dir = r'../data/cleaned'
reports_dir = r'../reports'

os.makedirs(cleaned_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)
print('Environment initialized.')

## Step 1: Load Raw Datasets
Load datasets safely using appropriate encoding handlers.

In [ ]:
def safe_read(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin1', low_memory=False)

qs_path = os.path.join(raw_dir, 'QS_Ranking', 'QS World University Rankings 2025 (Top global universities).csv')
the_path = os.path.join(raw_dir, 'World_Ranking', 'World University Rankings 2023.csv')
ed_path = os.path.join(raw_dir, 'archive (8)', 'edstats-csv-zip-32-mb-', 'EdStatsData.csv')
cntry_path = os.path.join(raw_dir, 'archive (8)', 'edstats-csv-zip-32-mb-', 'EdStatsCountry.csv')

df_qs = safe_read(qs_path)
df_the = safe_read(the_path)
df_ed = safe_read(ed_path)
df_cntry = safe_read(cntry_path)

print(f'QS 2025 Loaded: {df_qs.shape}')
print(f'THE 2023 Loaded: {df_the.shape}')
print(f'EdStats Data Loaded: {df_ed.shape}')
print(f'EdStats Country Loaded: {df_cntry.shape}')

## Step 2: Standardize Column Names & Strip Text Whitespace
Convert column names to consistent `lower_snake_case` and trim whitespace.

In [ ]:
# QS 2025 column renaming
col_map_qs = {
    'RANK_2025': 'rank_2025',
    'RANK_2024': 'rank_2024',
    'Institution_Name': 'university_name',
    'Location': 'country',
    'Region': 'region',
    'SIZE': 'institution_size',
    'FOCUS': 'subject_focus',
    'RES.': 'research_intensity',
    'STATUS': 'institution_status',
    'Academic_Reputation_Score': 'academic_reputation_score',
    'Academic_Reputation_Rank': 'academic_reputation_rank',
    'Employer_Reputation_Score': 'employer_reputation_score',
    'Employer_Reputation_Rank': 'employer_reputation_rank',
    'Faculty_Student_Score': 'faculty_student_score',
    'Faculty_Student_Rank': 'faculty_student_rank',
    'Citations_per_Faculty_Score': 'citations_per_faculty_score',
    'Citations_per_Faculty_Rank': 'citations_per_faculty_rank',
    'International_Faculty_Score': 'international_faculty_score',
    'International_Faculty_Rank': 'international_faculty_rank',
    'International_Students_Score': 'international_students_score',
    'International_Students_Rank': 'international_students_rank',
    'International_Research_Network_Score': 'international_research_network_score',
    'International_Research_Network_Rank': 'international_research_network_rank',
    'Employment_Outcomes_Score': 'employment_outcomes_score',
    'Employment_Outcomes_Rank': 'employment_outcomes_rank',
    'Sustainability_Score': 'sustainability_score',
    'Sustainability_Rank': 'sustainability_rank',
    'Overall_Score': 'overall_score'
}
df_qs.rename(columns=col_map_qs, inplace=True)

# THE 2023 column renaming
col_map_the = {
    'University Rank': 'world_rank',
    'Name of University': 'university_name',
    'Location': 'country',
    'No of student': 'num_students_raw',
    'No of student per staff': 'student_staff_ratio',
    'International Student': 'pct_international_students_raw',
    'Female:Male Ratio': 'female_male_ratio',
    'OverAll Score': 'overall_score',
    'Teaching Score': 'teaching_score',
    'Research Score': 'research_score',
    'Citations Score': 'citations_score',
    'Industry Income Score': 'industry_income_score',
    'International Outlook Score': 'international_outlook_score'
}
df_the.rename(columns=col_map_the, inplace=True)

# Strip text columns
for df in [df_qs, df_the]:
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.strip().replace({'nan': np.nan, 'NaN': np.nan, '': np.nan, '-': np.nan})

print('Columns renamed and string whitespace trimmed.')

## Step 3: Numeric Parsing & Clean Secondary Fields
Convert formatted string numbers (`42%`, `20,965`, `501-600`) into parsed numeric representations while retaining the original text for reference.

In [ ]:
# THE 2023 numeric parsing
df_the['num_students'] = pd.to_numeric(df_the['num_students_raw'].astype(str).str.replace(',', ''), errors='coerce')
df_the['pct_international_students'] = pd.to_numeric(df_the['pct_international_students_raw'].astype(str).str.replace('%', ''), errors='coerce')

def parse_rank(val):
    if pd.isna(val) or val == 'Reporter': return np.nan
    v = str(val).replace('=', '').replace('+', '').replace('–', '-').strip()
    if '-' in v:
        p = v.split('-')
        try: return (float(p[0]) + float(p[1])) / 2.0
        except: return np.nan
    m = re.search(r'(\d+)', v)
    return float(m.group(1)) if m else np.nan

df_the['world_rank_numeric'] = df_the['world_rank'].apply(parse_rank)
df_qs['rank_2025_numeric'] = df_qs['rank_2025'].apply(parse_rank)
df_qs['overall_score_numeric'] = pd.to_numeric(df_qs['overall_score'], errors='coerce')

print('Numeric parsing completed.')

## Step 4: Duplicate Row Deletion & Missing Value Inspection
Identify exact duplicate rows and remove confirmed duplicates.

In [ ]:
print('Exact duplicates in QS 2025:', df_qs.duplicated().sum())
print('Exact duplicates in THE 2023:', df_the.duplicated().sum())

# Drop duplicates from THE 2023
df_the.drop_duplicates(inplace=True)
print('Shape of THE 2023 after duplicate removal:', df_the.shape)

## Step 5: Export Cleaned Datasets
Save outputs into `data/cleaned/`.

In [ ]:
df_qs.to_csv(os.path.join(cleaned_dir, 'qs_2025_cleaned.csv'), index=False)
df_the.to_csv(os.path.join(cleaned_dir, 'wur_2023_cleaned.csv'), index=False)
df_the.to_csv(os.path.join(cleaned_dir, 'the_2024_cleaned.csv'), index=False)
df_ed.to_csv(os.path.join(cleaned_dir, 'world_bank_education_cleaned.csv'), index=False)
df_cntry.to_csv(os.path.join(cleaned_dir, 'world_bank_country_cleaned.csv'), index=False)

print('All cleaned datasets exported successfully.')